# Logistic Regression — Derivation from Maximum Likelihood

## Goal

This notebook derives logistic regression from first principles: why
sigmoid replaces the step function from the perceptron, how Maximum
Likelihood Estimation leads to the binary cross-entropy loss function,
and the full gradient derivation needed for gradient descent — since,
unlike linear regression, logistic regression has no closed-form
solution.

## The Model — Why Sigmoid, Not a Step Function?

$$ z = B_0 + B_1x_1 + \dots + B_nx_n, \qquad \hat y = \sigma(z) = \frac{1}{1+e^{-z}} $$

The original perceptron used a **step function** — output exactly $0$ or
$1$, with a sudden jump and no in-between values. This has a fatal
problem for gradient-based learning: a step function is not
differentiable at the jump, so there's no slope to follow toward better
parameters.

**Sigmoid** solves this by outputting a smooth value between $0$ and $1$,
differentiable everywhere — making gradient descent possible. $\hat y$ is
interpreted as the model's estimated **probability** that $y=1$.

## Deriving the Sigmoid's Derivative

$$ \sigma(z) = \frac{1}{1+e^{-z}} $$

Differentiating with respect to $z$ (chain rule, treating this as
$(1+e^{-z})^{-1}$):

$$ \frac{d\sigma}{dz} = -1\cdot(1+e^{-z})^{-2}\cdot(-e^{-z}) = \frac{e^{-z}}{(1+e^{-z})^2} $$

Splitting into two fractions:

$$ \frac{d\sigma}{dz} = \frac{1}{1+e^{-z}} \times \frac{e^{-z}}{1+e^{-z}} $$

Rewriting the second fraction (adding and subtracting $1$ in the
numerator):

$$ \frac{e^{-z}}{1+e^{-z}} = \frac{(1+e^{-z})-1}{1+e^{-z}} = 1-\sigma(z) $$

$$ \boxed{\frac{d\sigma}{dz} = \sigma(z)\big[1-\sigma(z)\big]} $$

**Why is this remarkable?** The derivative is expressed entirely in terms
of $\sigma(z)$ itself — once $\hat y_i = \sigma(z_i)$ is computed in the
forward pass, no extra exponential evaluation is needed to get its
derivative.

## Maximum Likelihood: One Data Point

We want the model to assign high probability to what was *actually*
observed. For one data point, the probability the model assigns to the
true label $y_i$ is:

$$ P(y_i|\hat y_i) = \hat y_i^{\,y_i}(1-\hat y_i)^{1-y_i} $$

**Why does this one formula work for both classes?** If $y_i=1$, this
becomes $\hat y_i^1(1-\hat y_i)^0=\hat y_i$ — the predicted probability of
class 1. If $y_i=0$, it becomes $\hat y_i^0(1-\hat y_i)^1=1-\hat y_i$ —
the predicted probability of class 0. The unused term always raises to
the power $0$ and vanishes, so one expression correctly covers both
cases without an if/else.

## Likelihood Across All Points, and Why We Take the Log

For all $n$ points to be simultaneously correct, multiply their
individual probabilities:

$$ L = \prod_{i=1}^{n}\hat y_i^{\,y_i}(1-\hat y_i)^{1-y_i} $$

**Why take the log?** Multiplying many probabilities (each between 0
and 1) shrinks toward zero extremely fast — for example, multiplying ten
values around $0.8$ together gives a number near $0.1$, and this gets
numerically unstable (risking underflow) as $n$ grows. Taking the log
converts this product into a **sum**, which is numerically stable and
far easier to differentiate term by term:

$$ \log L = \sum_{i=1}^{n}\Big[y_i\log\hat y_i + (1-y_i)\log(1-\hat y_i)\Big] $$

**Why negative log-likelihood?** Since each $\hat y_i$ and $(1-\hat y_i)$
is between 0 and 1, their logs are negative — so $\log L$ itself is
negative, and *maximizing* a negative quantity is the same as
*minimizing* its positive counterpart. Taking the negative keeps this
consistent with every other loss function in this project, where the
goal is always to minimize:

$$ E = -\sum_{i=1}^{n}\Big[y_i\log\hat y_i + (1-y_i)\log(1-\hat y_i)\Big] $$

This is the **binary cross-entropy loss**.

## Model Comparison with Scikit-Learn

The custom Logistic Regression implemented in this notebook updates its parameters using **Stochastic Gradient Descent (SGD)**, where the weights are updated after processing one training sample at a time.

Since Scikit-Learn's `SGDClassifier` also optimizes Logistic Regression using SGD when `loss="log_loss"` is specified, it provides the most appropriate implementation for validating the correctness of this custom model.

The comparison below evaluates both implementations using the same dataset and similar hyperparameters.

In [2]:
import numpy as np
class My_Logistic_Regression():
    def __init__ (self):
        self.intercept_=None
        self.coef_=None
        self.learning_rate=None
        self.epoches=None

    def sigmoid(self,z):
        return 1/(1+np.exp(-z))
        
    def fit(self,x_train,y_train,learning_rate=0.01,epoches=500):
        x_train=np.insert(x_train,0,1,axis=1)
        weights=np.zeros(x_train.shape[1])
        self.learning_rate=learning_rate
        self.epoches=epoches
        for i in range (self.epoches):
            indices=np.random.permutation(x_train.shape[0])
            for idx in indices:
                y_hat=self.sigmoid(np.dot(x_train[idx],weights))
                weights=weights+(self.learning_rate*(y_train[idx]-y_hat)*x_train[idx])

        self.intercept_=weights[0]
        self.coef_=weights[1:]
    def predict_proba(self, x_test):
            return self.sigmoid(self.intercept_ +np.dot(x_test, self.coef_))
    
    def predict(self,x_test):
        probs= self.sigmoid(self.intercept_ + np.dot(x_test,self.coef_))
        return (probs>=0.5).astype(int)

In [7]:
from sklearn.datasets import make_classification 
from sklearn.model_selection import train_test_split
x,y=make_classification(n_samples=300,n_features=3,n_informative=3,n_redundant=0,random_state=42)

x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)


In [13]:
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score,precision_score,recall_score
sk_model = SGDClassifier(loss="log_loss",penalty=None,learning_rate="constant",eta0=0.01,max_iter=1500,shuffle=True,random_state=42)

my_model=My_Logistic_Regression()

sk_model.fit(x_train,y_train)
my_model.fit(x_train,y_train,epoches=1500)

sk_pred=sk_model.predict(x_test)
my_pred=my_model.predict(x_test)

print("My Logistic Regression:")
print("Accuracy Score : ",accuracy_score(y_test,my_pred))
print("Precision : ",precision_score(y_test,my_pred))
print("Recall : ",recall_score(y_test,my_pred))

print("Sklearn stochastic Gradient Classifier")
print("Accuracy score : ",accuracy_score(y_test,sk_pred))
print("Precision : ", precision_score(y_test,sk_pred))
print("Recall : ",recall_score(y_test,sk_pred))

My Logistic Regression:
Accuracy Score :  0.85
Precision :  0.8214285714285714
Recall :  0.8518518518518519
Sklearn stochastic Gradient Classifier
Accuracy score :  0.85
Precision :  0.8214285714285714
Recall :  0.8518518518518519
